# Retraining Down Syndrome Screening (revisi BEEI #13183)

Notebook ini menjalankan ulang seluruh eksperimen secara **reproducible**:

| Masalah di naskah lama | Perbaikan di sini | Reviewer |
|---|---|---|
| Split tanpa seed, berbeda tiap notebook | Satu split ber-seed, disimpan ke CSV, dipakai semua model | B2 |
| Risiko gambar duplikat lintas train/test | Perceptual hash, gambar mirip dikelompokkan, satu grup selalu di subset yang sama | B1 |
| `BinaryCrossentropy(from_logits=True)` + sigmoid | Loss diperbaiki (`from_logits=False`) | - |
| Augmentasi agresif tanpa ablasi | Ablasi: `none` vs `original` vs `conservative` | B3 |
| Hanya pasangan DenseNet+Xception | Concat: D+X, D+I3, D+IR, X+IR, D+X+IR; soft voting semua kombinasi | B5 |
| Tanpa pembanding arsitektur modern | ConvNeXt-Tiny ditambahkan sebagai backbone tunggal | A2, C2 |
| Tanpa AUC/ROC/PR/CI/uji statistik | ROC, PR, AUC, bootstrap CI, Wilson CI, McNemar, threshold Youden dari validasi | B6 |
| Tanpa biaya komputasi | Parameter, ukuran, latensi CPU/GPU, ukuran TFLite int8 | C3 |
| Tanpa Grad-CAM | Grad-CAM per cabang pada TP/FP/FN | C1 |

**Persiapan:** buat folder `MyDrive/DS_Retraining`, lalu upload `ds_pipeline.py` dan `DownSyndrome.zip` ke sana.
Runtime: **GPU** (T4 cukup). Semua hasil tersimpan di Drive; kalau sesi putus, jalankan ulang dari atas, run yang sudah selesai otomatis dilewati.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install imagehash

import os, sys, json, itertools
BASE = '/content/drive/MyDrive/DS_Retraining'   # berisi ds_pipeline.py + DownSyndrome.zip
OUT = f'{BASE}/runs'
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, BASE)
if not os.path.isdir('/content/DownSyndrome'):
    !unzip -q "{BASE}/DownSyndrome.zip" -d /content
DATA = '/content/DownSyndrome'
SEED = 42

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
import ds_pipeline as dsp
dsp._selfcheck()
print(tf.__version__, tf.config.list_physical_devices('GPU'))

## 1. Deteksi duplikat dan split ber-seed (dibuat sekali, lalu dipakai ulang)

In [ ]:
split_csv = f'{OUT}/split_seed{SEED}.csv'
if not os.path.exists(split_csv):
    df, report = dsp.add_groups(dsp.list_images(DATA), max_dist=4)
    df['split'] = dsp.split_groups(df, SEED)
    df['rel'] = df.path.str.slice(len(DATA) + 1)
    df['phash'] = df.phash.map(lambda v: f'{int(v):016x}')
    df[['rel', 'y', 'phash', 'group', 'split']].to_csv(split_csv, index=False)
    json.dump(report, open(f'{OUT}/dedup_report.json', 'w'), indent=2)
df = pd.read_csv(split_csv)
df['path'] = DATA + '/' + df.rel
split = df.split
print(json.load(open(f'{OUT}/dedup_report.json')))
print(pd.crosstab(df.split, df.y.map({1: 'DS', 0: 'healthy'}), margins=True))

## 2. Ablasi augmentasi (DenseNet-201)
Kebijakan dipilih berdasarkan **AUC validasi**, bukan test, supaya test set tetap bersih.

In [ ]:
POLICIES = ['none', 'original', 'conservative']
for pol in POLICIES:
    dsp.train_run(['DenseNet201'], df, split, OUT, policy=pol, seed=SEED)

abl_tags = [dsp.run_name(['DenseNet201'], p) for p in POLICIES]
abl = dsp.collect_preds(OUT, abl_tags)
from sklearn.metrics import roc_auc_score
val = abl[abl.split == 'val']
val_auc = {p: roc_auc_score(val.y, val[t]) for p, t in zip(POLICIES, abl_tags)}
POLICY = max(val_auc, key=val_auc.get)
print('val AUC:', val_auc, '-> dipakai:', POLICY)
abl_sum = dsp.summarize(abl, abl_tags)
abl_sum.to_csv(f'{OUT}/ablation_augmentation.csv', index=False)
abl_sum[['config', 'threshold_rule', 'accuracy', 'sensitivity', 'specificity', 'auc', 'auc_lo', 'auc_hi']].round(3)

## 3. Training semua konfigurasi (resumable)
Ensemble 3 backbone memakai batch 16 agar muat di memori T4 (dicatat di `meta.json`).

In [ ]:
SINGLES = [[n] for n in dsp.BACKBONES]
FUSIONS = [['DenseNet201', 'Xception'], ['DenseNet201', 'InceptionV3'],
           ['DenseNet201', 'InceptionResNetV2'], ['Xception', 'InceptionResNetV2'],
           ['DenseNet201', 'Xception', 'InceptionResNetV2']]
for names in SINGLES + FUSIONS:
    dsp.train_run(names, df, split, OUT, policy=POLICY, seed=SEED, bs=16 if len(names) == 3 else 32)

## 4. Soft voting, ringkasan metrik, dan CI

In [ ]:
tags = [dsp.run_name(n, POLICY) for n in SINGLES + FUSIONS]
preds = dsp.collect_preds(OUT, tags)
single_tags = [dsp.run_name(n, POLICY) for n in SINGLES]
for k in (2, 3, 4):
    for combo in itertools.combinations(single_tags, k):
        preds['SV(' + '+'.join(t.split('__')[0] for t in combo) + ')'] = dsp.soft_vote(preds, list(combo))
preds.to_csv(f'{OUT}/all_predictions.csv', index=False)

configs = [c for c in preds.columns if c not in ('path', 'y', 'split')]
summary = dsp.summarize(preds, configs)
summary.to_csv(f'{OUT}/summary_test.csv', index=False)
cols = ['config', 'threshold_rule', 'threshold', 'TP', 'FN', 'FP', 'TN', 'accuracy', 'precision',
        'sensitivity', 'specificity', 'npv', 'f1', 'auc', 'auc_lo', 'auc_hi', 'ap', 'brier']
summary[summary.threshold_rule == '0.5'][cols].sort_values('auc', ascending=False).round(3)

## 5. Uji berpasangan (McNemar exact + selisih AUC) terhadap concat DenseNet+Xception

In [ ]:
REF = dsp.run_name(['DenseNet201', 'Xception'], POLICY)
test = preds[preds.split == 'test']
rows = []
for c in configs:
    if c == REF:
        continue
    b, cc, p = dsp.mcnemar_exact(test.y, test[REF], test[c])
    d, (lo, hi) = dsp.paired_auc_diff(test.y, test[REF], test[c])
    rows.append({'vs': c, 'ref_only_correct': b, 'other_only_correct': cc, 'mcnemar_p': p,
                 'auc_diff_ref_minus_other': d, 'diff_lo': lo, 'diff_hi': hi})
paired = pd.DataFrame(rows)
from statsmodels.stats.multitest import multipletests
paired['mcnemar_p_holm'] = multipletests(paired.mcnemar_p, method='holm')[1]
paired.to_csv(f'{OUT}/paired_tests_vs_{REF}.csv', index=False)
paired.round(4)

## 6. Kurva ROC & Precision-Recall, confusion matrix (label benar: DS = positif)

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
MAIN = single_tags + [REF, 'SV(D+X)']
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for c in MAIN:
    fpr, tpr, _ = roc_curve(test.y, test[c]); pr, rc, _ = precision_recall_curve(test.y, test[c])
    auc = summary.query('config == @c and threshold_rule == "0.5"').auc.iloc[0]
    ax[0].plot(fpr, tpr, label=f'{c} (AUC {auc:.3f})'); ax[1].plot(rc, pr, label=c)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(xlabel='1 - specificity', ylabel='Sensitivity', title='ROC (test)')
ax[1].set(xlabel='Recall (sensitivity)', ylabel='Precision', title='Precision-recall (test)')
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{OUT}/roc_pr_test.png', dpi=300); plt.show()

fig, axs = plt.subplots(1, len(MAIN), figsize=(3.2 * len(MAIN), 3.2))
for a, c in zip(axs, MAIN):
    ConfusionMatrixDisplay.from_predictions(test.y.map({1: 'DS', 0: 'Healthy'}),
        np.where(test[c] >= 0.5, 'DS', 'Healthy'), labels=['DS', 'Healthy'], ax=a, colorbar=False)
    a.set_title(c, fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT}/confusion_matrices_test.png', dpi=300); plt.show()

## 7. Biaya komputasi (parameter, ukuran, latensi batch-1, TFLite int8)

In [ ]:
comp = []
for names in SINGLES + FUSIONS:
    m = dsp.load_model(names, f'{OUT}/{dsp.run_name(names, POLICY)}')
    r = dsp.benchmark(m, tflite_dir=OUT)
    comp.append({'config': dsp.run_name(names, POLICY), **r})
    tf.keras.backend.clear_session()
comp = pd.DataFrame(comp)
comp['gpu'] = str(tf.config.list_physical_devices('GPU'))
!nvidia-smi --query-gpu=name --format=csv,noheader
comp.to_csv(f'{OUT}/compute.csv', index=False)
comp.round(2)

## 8. Grad-CAM pada concat DenseNet+Xception (TP, FP, FN)

In [ ]:
NAMES = ['DenseNet201', 'Xception']
m = dsp.load_model(NAMES, f'{OUT}/{REF}')
t = test.copy(); t['p'] = t[REF]
picks = pd.concat([t[(t.y == 1) & (t.p >= .5)].nlargest(3, 'p').assign(kind='TP'),
                   t[(t.y == 0) & (t.p >= .5)].nlargest(4, 'p').assign(kind='FP'),
                   t[(t.y == 1) & (t.p < .5)].nsmallest(3, 'p').assign(kind='FN')])
fig, axs = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
for row, (_, r) in zip(axs, picks.iterrows()):
    img = tf.image.resize(tf.io.decode_image(tf.io.read_file(r.path), channels=3, expand_animations=False),
                          (224, 224)).numpy().astype('uint8')
    row[0].imshow(img); row[0].set_title(f"{r.kind}  p(DS)={r.p:.2f}", fontsize=9)
    for a, br in zip(row[1:], NAMES):
        cam, _ = dsp.gradcam(m, NAMES, img, br)
        a.imshow(img); a.imshow(cam, cmap='jet', alpha=0.4); a.set_title(f'Grad-CAM {br}', fontsize=9)
    for a in row: a.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/gradcam_concat_DX.png', dpi=200); plt.show()
picks[['path', 'kind', 'p']].to_csv(f'{OUT}/gradcam_cases.csv', index=False)

## 9. (Opsional) Repeated stratified group 5-fold CV
Mahal: 5 fold x konfigurasi. Default hanya 3 konfigurasi utama dan 1 repeat. Set `RUN_CV = True` untuk menjalankan.

In [ ]:
RUN_CV, K, REPEATS = False, 5, 1
CV_CONFIGS = [['DenseNet201'], ['InceptionResNetV2'], ['DenseNet201', 'Xception']]
if RUN_CV:
    rows = []
    for r, f, sp in dsp.cv_folds(df, k=K, repeats=REPEATS, seed=SEED):
        for names in CV_CONFIGS:
            tag = f'{dsp.run_name(names, POLICY)}_r{r}f{f}'
            p = dsp.train_run(names, df, sp, f'{OUT}/cv', policy=POLICY, seed=SEED, tag=tag)
            te = p[p.split == 'test']
            rows.append({'config': dsp.run_name(names, POLICY), 'repeat': r, 'fold': f,
                         **dsp.metrics(te.y, te[tag], 0.5)})
    cv = pd.DataFrame(rows); cv.to_csv(f'{OUT}/cv_folds.csv', index=False)
    display(cv.groupby('config')[['accuracy', 'sensitivity', 'specificity', 'f1', 'auc']].agg(['mean', 'std']).round(3))

## 10. File yang perlu dikirim balik untuk update naskah
`dedup_report.json`, `split_seed42.csv`, `ablation_augmentation.csv`, `summary_test.csv`, `paired_tests_vs_*.csv`,
`compute.csv`, `roc_pr_test.png`, `confusion_matrices_test.png`, `gradcam_concat_DX.png`, `gradcam_cases.csv`,
semua `meta.json`, dan `cv_folds.csv` kalau CV dijalankan.

In [ ]:
import shutil, glob
pack = '/content/results_for_paper'
os.makedirs(pack, exist_ok=True)
for pat in ['*.csv', '*.json', '*.png', '*/meta.json', '*/history.csv']:
    for f in glob.glob(f'{OUT}/{pat}'):
        dst = os.path.join(pack, os.path.relpath(f, OUT).replace('/', '__'))
        shutil.copy(f, dst)
shutil.make_archive(f'{BASE}/results_for_paper', 'zip', pack)
print('Tersimpan:', f'{BASE}/results_for_paper.zip')